# 🥈 NYC Yellow Taxi Analytics — Bronze to Silver Refinement
---
### 👤 Developer Profile
* **Name:** `cacciari@gmail.com`
* **Contact:** [cacciari@gmail.com](mailto:cacciari@gmail.com)
* **LinkedIn:** [linkedin.com/in/cacciari](https://linkedin.com/in/cacciari)

### 📋 Module Overview
* **File Name:** `03_bronze_to_silver`
* **Target Layer:** `Silver Layer (Single Source of Truth)`
* **Short Description:** Core Data Quality engine applying specialized validation gates, FinOps filter constraints, record deduplication, and primitive type harmonization. Drops analytical noise columns and outputs sanitized data ready for advanced statistical queries.

In [0]:
%skip
%pip install datacompy
%pip install datacompy[spark]

In [0]:
%restart_python

#### Imports

In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name, col, lit, first, date_add
from pyspark.sql.types import StructType, StructField, StringType
from typing import Dict, List, Any
from constants import BRONZE_TABLE, SILVER_TABLE, TARGET_RAW_COLUMNS, AUDIT_COLUMNS
from modules.data_ingestion import *


#### Functions

#### Pipeline configuration

In [0]:
AUDIT_COLUMNS["AUD_ING_BRZ_TO_SLV_SCRIPT"] = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get().split('/')[-1]

# WIDGET
dbutils.widgets.text("SILVER_TABLE", SILVER_TABLE)


#### Pre-Silver Ingestion Checks
* Show ingestion parameters

In [0]:
print("Parameter Set:")
print(f"\tBronze Table: {BRONZE_TABLE}")
print(f"\tSilver Table: {SILVER_TABLE}")

print (f"Starting Transformation from {BRONZE_TABLE} to {SILVER_TABLE}")

try:
    print(f"\nAcquiring Data...:")
    df_bronze = spark.table(BRONZE_TABLE)
    print(f"\t{BRONZE_TABLE} acquired. Total records: {df_bronze.count()}")

except:
    print(f"\t{BRONZE_TABLE} not found.")
    raise Exception(f"\t{BRONZE_TABLE} not found.")

df_silver = df_bronze                 

# Passenger Count
rows_before = df_silver.count()
df_silver = df_silver.filter(col("passenger_count") > 0)
rows_after = df_silver.count()
print(f"\tZeroed and negative passengers removed: {rows_before - rows_after}. Remaining: {rows_after}")

# Fare Amount
rows_before = rows_after
df_silver = df_silver.filter(col("fare_amount") > 0)
rows_after = df_silver.count()
print(f"\tZeroed and negative fare amounts (base fare) removed: {rows_before - rows_after}. Remaining: {rows_after}")

# Total Amount
rows_before = rows_after
df_silver = df_silver.filter(col("total_amount") > 0)
rows_after = df_silver.count()
print(f"\tZeroed and negative total amounts removed: {rows_before - rows_after}. Remaining: {rows_after}")

# Total vs Fare
rows_before = rows_after
df_silver = df_silver.filter(col("total_amount") >= col("fare_amount"))
rows_after = df_silver.count()
print(f"\tInconsistent Total < Fare anomalies removed: {rows_before - rows_after}. Remaining: {rows_after}")

# Trip Distance
rows_before = rows_after
df_silver = df_silver.filter(col("trip_distance") > 0)
rows_after = df_silver.count()
print(f"\tZeroed and negative trip distances removed: {rows_before - rows_after}. Remaining: {rows_after}")

# NYC Codes and Regions
# Codes
rows_before = rows_after
df_silver = df_silver.filter(col("RatecodeID").between(1, 6))
rows_after = df_silver.count()
print(f"\tInvalid RatecodeID removed: {rows_before - rows_after}. Remaining: {rows_after}")

# Regions Pickup
rows_before = rows_after
df_silver = df_silver.filter(col("PULocationID").between(1, 263))
rows_after = df_silver.count()
print(f"\tInvalid PULocationID removed: {rows_before - rows_after}. Remaining: {rows_after}")

# Regions DropOff
rows_before = rows_after
df_silver = df_silver.filter(col("DOLocationID").between(1, 263))
rows_after = df_silver.count()
print(f"\tInvalid DOLocationID removed: {rows_before - rows_after}. Remaining: {rows_after}")

# Chronological consistency + Limit duration
rows_before = rows_after
df_silver = df_silver.filter(col("tpep_pickup_datetime") < col("tpep_dropoff_datetime"))
rows_after = df_silver.count()
print(f"\tInconsistent pickup > dropoff timestamps removed: {rows_before - rows_after}. Remaining: {rows_after}")

# Travel durations too long
rows_before = rows_after
df_silver = df_silver.filter(col("tpep_dropoff_datetime") <= date_add(col("tpep_pickup_datetime"),1))
rows_after = df_silver.count()
print(f"\tTravel durations > 1 day removed: {rows_before - rows_after}. Remaining: {rows_after}")

# Pickup time scope : 2023: January to May
rows_before = rows_after
df_silver = df_silver.filter(col("tpep_pickup_datetime").between(lit("2023-01-01"), lit("2023-05-31")))
rows_after = df_silver.count()
print(f"\tPickup time scope after outside January to May removed: {rows_before - rows_after}. Remaining: {rows_after}")


## Silver Dedups
df_silver = df_silver.dropDuplicates()
print(f"\tDuplicates removed: {rows_after - df_silver.count()}. Remaining: {df_silver.count()}")

In [0]:
## Ingest to Silver
AUDIT_COLUMNS["CURRENT_SCRIPT"] = AUDIT_COLUMNS["AUD_ING_BRZ_TO_SLV_SCRIPT"]
if ingest_data({"silver_processed:":df_silver}, SILVER_TABLE, AUDIT_COLUMNS ):
    print("Data successfully ingested")
else:
    print("Data ingestion failed")